<a href="https://colab.research.google.com/github/geeta-gwalior/Gemma-with-RAG-A-Coffee-Shop-Chatbot/blob/main/Gemma_with_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U "transformers>=4.50.0" accelerate torch safetensors bitsandbytes
!pip install -U sentence-transformers faiss-cpu


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.2 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.4
    Uninstalling tokenizers-0.21.4:
      Successfully uninstalled tokenizers-0.21.4
  Attempting uninstall: transformers
    Found existing installation: transformers 4.55.4
    Uninstalling transformers-4.55.4:
      Successfully uninstalled transformers-4.55.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 62.2 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "google/gemma-3-1b-it"   # 4B fits most Colab GPUs

tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

chat_history = []

def chat(query):
    global chat_history
    chat_history.append({"role":"user","content":query})
    prompt = tok.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
    reply = tok.decode(outputs[0], skip_special_tokens=True)

    # extract only assistant's last message
    reply = reply.split(query)[-1].strip()
    chat_history.append({"role":"assistant","content":reply})
    return reply


In [ ]:
while True:
    user_inp = input("You: ")
    if user_inp.lower() in ["quit","exit","stop"]:
        break
    response = chat(user_inp)
    print("Gemma:", response)


You: hi
Gemma: ng new?** (I can provide information on many subjects)

Let me know!
You: can you explain about delhi
Gemma: model
Okay, let's dive into Delhi! Here's a breakdown of what Delhi is, covering various aspects:

**1. What is Delhi?**

Delhi is the capital city of India and the most populous city in the country. It’s a sprawling metropolis, a historical powerhouse, and a vibrant cultural hub. It's often called the “City of a Thousand Hills” due to its hilly terrain and historical significance.

**2. History:**

* **Ancient Roots:** Delhi's history stretches back millennia, with evidence of settlements dating back to the Indus Valley Civilization. It was a crucial trading center for centuries.
* **Mughal Empire:** The city flourished as the capital of the Mughal Empire (16th-18th centuries), a period of immense cultural and architectural influence.  Key historical sites include the Red Fort, Humayun's Tomb, and Qutub Minar.
* **British Rule:**  Delhi became a British
You: stop

In [ ]:
import pandas as pd

# Load your uploaded JSONL dataset
df = pd.read_json("/content/coffeeworld_dataset.jsonl", lines=True)
display(df.head())

,input,output
0,What is CoffeeWorld?,CoffeeWorld is an Indian cafe that offers uniq...
1,Where is CoffeeWorld located?,CoffeeWorld is located in India and offers a c...
2,What makes CoffeeWorld unique?,We blend traditional Indian flavors with globa...
3,Do you have South Indian filter coffee?,"Yes! At CoffeeWorld, our South Indian filter c..."
4,Do you serve cold coffee?,Yes! We serve a variety of cold coffees includ...


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

# Use a small fast embedding model
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Store the input questions and outputs
questions = df["input"].tolist()
answers = df["output"].tolist()

# Encode the input questions
X = embedder.encode(questions, normalize_embeddings=True)

# Build FAISS index
index = faiss.IndexFlatIP(X.shape[1])
index.add(X)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
chat_history = []

def coffee_chat(query, k=1):
    global chat_history

    # Find best-matching FAQ question
    qv = embedder.encode([query], normalize_embeddings=True)
    D, I = index.search(qv, k)

    # Retrieve the most relevant answer
    answer = answers[I[0][0]]

    # Add to history
    chat_history.append({"user": query, "bot": answer})

    return answer


In [ ]:
while True:
    user_inp = input("You: ")
    if user_inp.lower() in ["quit","exit","stop"]:
        print("CoffeeBot: Goodbye 👋")
        break
    response = coffee_chat(user_inp)
    print("CoffeeBot:", response)


You: hi
CoffeeBot: Hi there! Welcome to CoffeeWorld. How can I help you today?
You: What is CoffeeWorld?
CoffeeBot: CoffeeWorld is an Indian cafe that offers unique coffees inspired by South Indian and world flavors.
You: what is spacial about CoffeeWorld?
CoffeeBot: CoffeeWorld is an Indian cafe that offers unique coffees inspired by South Indian and world flavors.
You: what is best coffee in coffeeworld?
CoffeeBot: Our two best sellers are the 'Royal South Indian Filter Coffee' and the 'World Fusion Cold Brew'.
You: which coffee i should try in winter?
CoffeeBot: For winter, try our Royal South Indian Filter Coffee or Masala Coffee.
You: give me coffeeworld menu?
CoffeeBot: CoffeeWorld is an Indian cafe that offers unique coffees inspired by South Indian and world flavors.
You: how many coffee are here?
CoffeeBot: Our popular coffees include Royal South Indian Filter Coffee and World Fusion Cold Brew.
You: how many coffee you serve?
CoffeeBot: We serve Indian filter coffee, masala co